In [ ]:
import numpy as np

# run the jupyter for each of the belows

In [ ]:
# data = np.load("/storage_common/nobilm/backmapping_pots_model/datasets/active/with_hs/without_hs/backmapping_dataset.npz") 
# data = np.load('/storage_common/nobilm/backmapping_pots_model/datasets/inactive/whs/without_hs/backmapping_dataset.npz')
data = np.load('/storage_common/nobilm/backmapping_pots_model/datasets/pas/with_hs/without_hs/backmapping_dataset.npz')
data = dict(data)
# [k for k in data.keys()]

## fix atom_resids by shifting it down by 2 since it was starting at 2 and we like at 0 s.t. avoid any mistake when indexing

In [ ]:
new_atom_resids = data['atom_resids'] - 2; new_atom_resids
data['atom_resids'] = new_atom_resids; new_atom_resids
seq = 'SSVYITVELAIAVLAILGNVLVCWAVWLNSNLQNVTNYFVVSLAAADIAVGVLAIPFAITISTGFCAACHGCLFIACFVLVLTQSSIFSLLAIAIDRYIAIRIPLRYNGLVTGTRAKGIIAICWVLSFAIGLTPMLGWNNCGQPKEGKNHSQGCGEGQVACLFEDVVPMNYMVYFNFFACVLVPLLLMLGVYLRIFLAARRQLKQMESQPLPGERARSTLQKEVHAAKSLAIIVGLFALCWLPLHIINCFTFFCPDCSHAPLWLMYLAIVLSHTNSVVNPFIYAYRIREFRQTFRKIIRS'
res_idx = list(set(data['atom_residue_index'].tolist()))
res_idx.sort()
data['res_idx'] = res_idx
# [k for k in data.keys()], len(seq)
# out:
# (['trajectory',
#   'atom_resids',
#   'atom_names',
#   'atom_residue_index',
#   'residue_keys',
#   'residue_cluster_ids',
#   'residue_cluster_counts',
#   'frame_indices',
#   'frame_state_ids',
#   'state_id',
#   'sample_id',
#   'dihedrals',
#   'dihedral_keys',
#   'dihedral_atom_indices',
#   'dihedral_mask',
#   'res_idx'],
#  300)

## build the mapping to global_id

In [ ]:
# create a list where each element is a tuple of (res_idx, residue_cluster_count) for that res_idx, since res_idx is constant across active/inactive/pas - since we are using only 1 protein -
# then we can build the mapping to global_id by iterating through the list residue_cluster_count assigning/creating a global_id for each cluster in that range(residue_cluster_count)

# so to do the mapping of clusters sampled the pots model we need to load up one of the active/inactive/pas npz files, get the res_idx and residue_cluster_counts, build the mapping to global_id, 
# s.t. then we can use that mapping to convert the cluster ids sampled by the pots model to global cluster ids that we can then use for conditioning the folding model @ inference time

In [ ]:
l = list(zip(data['res_idx'], data['residue_cluster_counts'])) 

# build the res_idx_to_glob_cluster_ids:
res_idx_to_glob_cluster_ids = {}
global_id = 0
for res_id, clu_count in l:
    for clu_global_idx in range(clu_count):
        res_idx_to_glob_cluster_ids[(res_id, clu_global_idx)] = global_id
        global_id += 1

# res_idx_to_glob_cluster_ids[(299, 17)] # 1682; 1682*768 = nn.embeddings 1'291'776

In [ ]:
# build the actual data
res_idx_and_glob_cluster_id = []
for frame in data['residue_cluster_ids']:
    for_this_frame = []
    for res_idx, cluster_id in enumerate(frame):
        glob_cluster_id = res_idx_to_glob_cluster_ids[(res_idx, cluster_id)]
        # for_this_frame.append((res_idx, glob_cluster_id))
        for_this_frame.append(glob_cluster_id)
    res_idx_and_glob_cluster_id.append(for_this_frame)

In [ ]:
data['res_idx_and_glob_cluster_id_per_frame'] = np.array(res_idx_and_glob_cluster_id) # (49996, 300)
data['atom_idx_and_glob_cluster_id_per_frame'] = data['res_idx_and_glob_cluster_id_per_frame'][:, new_atom_resids] # (49996, 2338)

# check the saved data

In [ ]:
# path = '/home/nobilm@usi.ch/ml-simplefold/test_new_data_with_clusters/pas_without_hs.npz'
# np.savez_compressed(path, **data)

# data_new = dict(np.load(path))
# for k,v in data_new.items(): print(k, v.shape)

# then go to cmds_after_writing_global_clustering_data.txt

# Cast new clusters labels sampled by pots model to global clusters
In #build the mapping to global_id above we create a list where each element is a tuple of (res_idx, residue_cluster_count) for that res_idx, since res_idx is constant across active/inactive/pas - since we are using only 1 protein - then we can build the mapping to global_id by iterating through the list residue_cluster_count assigning/creating a global_id for each cluster in that range(residue_cluster_count).
So to do the mapping of clusters sampled the pots model we need to load up one of the active/inactive/pas npz files, get the res_idx and residue_cluster_counts, build the mapping to global_id, 
s.t. then we can use that mapping to convert the cluster ids sampled by the pots model to global cluster ids that we can then use for conditioning the folding model @ inference time.

In [ ]:
npz_path_pots_samples = '/storage_common/angiod/phase-data/projects/a2a/systems/a2a/clusters/cb3c3cae-5316-47db-8fbb-0567d5f0f75b/samples/a1d26e1b-cf9e-43af-a7f6-b66bd026505a/sample.npz'
new_samples = np.load(npz_path_pots_samples)
new_samples = dict(new_samples)
# for k,v in new_samples.items(): print(k, v.shape) # labels (n_samples, cluster_id_per_residue: 300)

In [ ]:
new_samples['atom_idx_and_glob_cluster_id_per_frame'] = new_samples['labels'][:, data['atom_resids']]
new_samples['atom_idx_and_glob_cluster_id_per_frame'].shape # (n_samples, n_atoms_with_global_clusters) = (1000, 2338)

In [ ]:
save_path = '/storage_common/nobilm/backmapping_pots_model/pots_samples/sample.npz'
new_samples['og_npz_path'] = npz_path_pots_samples
np.savez_compressed(save_path, **new_samples)

In [ ]:
data_new = dict(np.load(save_path))
for k,v in data_new.items(): print(k, v.shape)
data_new['og_npz_path']